**Información previa:**

Este notebook carga los datos del fichero reign_data.xlsx en la base de datos *reign* en postgres. 

Los pasos previos para poder ejecutar el notebook son:
* A. Crear el contenedor docker con la base de datos postgres
* B. Crear la estructura de carpetas para incluír el fichero reign_data.xlsx
* C. Revisar los datos de conexión de la base de datos (actualmente con los mismos que el fichero .yaml)
* D. Ejecutar el notebook (!imporante, instala python si no lo tienes instalado)

A. *Creación de contenedor*

Para crear la base de datos debéis ejecutar el fichero .yaml de la siguiente manera:
1. Guardar el archivo como postgres_compose.yaml 
2. Navegar hasta el directorio desde la cmd
3. Ejecutar docker compose -f postgres_compose.yaml up -d

B. *Estructura de carpetas*

La estructura debe ser "data/reign_data.xlsx" o puedes cambiar la ruta de la variable *excel_path* de más abajo.

C. *Revisar los datos de conexión*

Actualmente los mismos indicados en el fichero .yaml pero debes revisar PGHOST, PGPORT, PGUSER, PGPASSWORD y PGDATABASE | db_name. Recuerda que si es un docker en local el host es "localhost" y si es la base de datos postgres el puerto por defecto es 5432.

D. *Ejecuta el notebook*

Asegurate que tienes el contenedor docker levantado, que la base de datos es accesible, que tienes python instalado y que la ruta y nombre del fichero son correcto.

*! Importante* , revisa el mensaje de la ejecucuón de la función create_structure_from_excel() y comprueba las validaciones de registros insertados.

Librerias necesarias para la ejecucón

In [120]:
pip install pandas psycopg2-binary openpyxl

Note: you may need to restart the kernel to use updated packages.


In [121]:
# librerías necesarias para trabajar con postgres y excel

import os
import re
import unicodedata
import pandas as pd
import psycopg2
from psycopg2 import sql
from datetime import datetime, timezone


Conexión a la base de datos y creación de estructura


In [122]:
PGHOST='localhost'
PGPORT=5432
PGUSER='root'
PGPASSWORD=1234
PGDATABASE='reign'

db_name = "reign"

DB_PREFIX = "reign_"            # prefijo de las BBDD por pestaña


In [123]:
# -------------------------------------------------------------------------------
# Conexión y test conexión
# -------------------------------------------------------------------------------
def connect(dbname: str):
    """Devuelve una conexión psycopg2 a la base de datos indicada."""
    return psycopg2.connect(
        host=PGHOST,
        port=PGPORT,
        user=PGUSER,
        password=PGPASSWORD,
        dbname=dbname
    )

def test_connection(dbname: str):
    try:
        conn = connect(dbname)   # ← aquí usamos tu función
        conn.close()
        return f"✅ Conexión exitosa a la base de datos '{dbname}'"
    except Exception as e:
        return f"❌ Error al conectar a la base de datos '{dbname}':\n{e}"


In [124]:
#Probar conexión 

print(test_connection("postgres"))

✅ Conexión exitosa a la base de datos 'postgres'


Lectura de fichero excel

In [125]:
#Excel path

excel_path = "data/reign_data.xlsx"

In [126]:
# Listado de pestaña de excel

def mostrar_pestanas_excel(excel_path):
    
    xls = pd.ExcelFile(excel_path, engine="openpyxl")
    print("Pestañas encontradas:")
    for nombre in xls.sheet_names:
        print(" -", nombre)

# Ejemplo de uso:
mostrar_pestanas_excel(excel_path)

Pestañas encontradas:
 - products_full
 - product_family
 - product_class
 - product_line
 - sales_rep
 - team_managers
 - stores
 - customers
 - all_sales


In [127]:
# -*- coding: utf-8 -*-
# Estructura de BBDD a partir de un Excel:
# - Usa mostrar_pestanas_excel() para detectar pestañas
# - Una base de datos por pestaña (prefijo configurable)
# - Borra la tabla si existe y la crea basándose en el contenido de la hoja
# - Mensajes de estado detallados

# ------------------------------------------------------------------------------------
# Función: mostrar_pestanas_excel (imprime y devuelve lista)
# ------------------------------------------------------------------------------------
def mostrar_pestanas_excel(ruta_excel: str, imprimir: bool = True) -> list[str]:
    """
    Devuelve la lista de pestañas (hojas) de un Excel y las imprime si imprimir=True.
    """
    xls = pd.ExcelFile(ruta_excel, engine="openpyxl")
    hojas = xls.sheet_names
    if imprimir:
        print("Pestañas encontradas:")
        for nombre in hojas:
            print(" -", nombre)
    return hojas

# ------------------------------------------------------------------------------------
# Utilidades de nombres y tipos
# ------------------------------------------------------------------------------------
def sanitize_identifier(name: str) -> str:
    """
    Convierte un nombre (hoja/columna) en identificador seguro para Postgres:
    - minúsculas, sin acentos; no alfanumérico -> '_'
    - colapsa múltiples '_' y recorta extremos
    """
    if name is None:
        name = "unnamed"
    name = "".join(c for c in unicodedata.normalize("NFKD", name) if not unicodedata.combining(c))
    name = re.sub(r"\W+", "_", name.lower())
    name = re.sub(r"_+", "_", name).strip("_")
    return name or "unnamed"

def pg_type_from_series(s: pd.Series) -> str:
    """
    Deducción robusta de tipos Postgres:
      - BOOLEAN si el contenido es y/n/true/false/1/0
      - INTEGER o BIGINT (si excede 32 bits)
      - DOUBLE PRECISION para floats
      - TIMESTAMP para fechas
      - TEXT para el resto
    """
    if s.dtype == "object":
        lowered = s.dropna().astype(str).str.strip().str.lower()
        if not lowered.empty and lowered.isin({"y","n","true","false","yes","no","t","f","1","0"}).all():
            return "BOOLEAN"
    if pd.api.types.is_bool_dtype(s):
        return "BOOLEAN"
    if pd.api.types.is_integer_dtype(s):
        max_abs = s.dropna().abs().max() if len(s.dropna()) else 0
        return "BIGINT" if pd.notna(max_abs) and max_abs and max_abs > 2_147_483_647 else "INTEGER"
    if pd.api.types.is_float_dtype(s):
        return "DOUBLE PRECISION"
    if pd.api.types.is_datetime64_any_dtype(s):
        return "TIMESTAMP"
    return "TEXT"

def normalize_boolean_column(series: pd.Series) -> pd.Series:
    """Convierte 'y/n/true/false/1/0' a booleanos; otros valores -> NULL."""
    def to_bool(x):
        if pd.isna(x): return None
        xs = str(x).strip().lower()
        if xs in ("y","yes","true","t","1"): return True
        if xs in ("n","no","false","f","0"): return False
        return None
    return series.map(to_bool)

def read_sheet_resilient(excel_path: str, sheet_name: str) -> pd.DataFrame:
    """
    Lectura robusta:
      - Intenta header=0
      - Si detecta '## título' en la primera celda, reintenta con header=1
      - Elimina filas/columnas totalmente vacías
    """
    df = pd.read_excel(excel_path, sheet_name=sheet_name, engine="openpyxl", header=0)
    if df.shape[1] == 1 and isinstance(df.columns[0], str) and df.columns[0].startswith("##"):
        df = pd.read_excel(excel_path, sheet_name=sheet_name, engine="openpyxl", header=1)
    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all")
    return df



# ------------------------------------------------------------------------------------
# 1) Crear tabla desde DataFrame (DROP + CREATE) en la BBDD TARGET_DB
# ------------------------------------------------------------------------------------
def create_table_from_df(conn, table_name: str, df: pd.DataFrame,
                         drop_before_create: bool = True,
                         cascade: bool = True,
                         ) -> pd.DataFrame:
    """
    Crea la tabla a partir del DataFrame:
      - Muestra nombre de tabla
      - Muestra columnas (normalizadas) y tipos deducidos
      - Muestra un listado de valores que se van a insertar (preview_rows primeras filas)
      - BORRA la tabla si existe
      - Deducción de tipos y CREATE TABLE

    Retorna el DataFrame ya normalizado (columnas y booleanos) para reutilizarlo en la inserción.
    """
    cur = conn.cursor()

    # 0) Nombre de la tabla
    print(f"==> Tabla objetivo: {table_name}")

    # 1) Borrar tabla si existe
    if drop_before_create:
        print(f"   → Borrando tabla si existe: {table_name} ...")
        drop_sql = sql.SQL("DROP TABLE IF EXISTS {} {}").format(
            sql.Identifier(table_name),
            sql.SQL("CASCADE;") if cascade else sql.SQL(";")
        )
        cur.execute(drop_sql)
        conn.commit()
        print(f"   ✓ Tabla eliminada (si existía): {table_name}")

    # 2) Normalizar nombres de columna
    df = df.rename(columns={c: sanitize_identifier(str(c)) for c in df.columns})

    # 3) Convertir posibles columnas booleanas
    for col in df.columns:
        if pg_type_from_series(df[col]) == "BOOLEAN" and df[col].dtype != bool:
            df[col] = normalize_boolean_column(df[col])

    # 4) Mostrar columnas y tipos deducidos
    #print("   → Columnas y tipos deducidos:")
    col_types = {col: pg_type_from_series(df[col]) for col in df.columns}
    #for col, typ in col_types.items(): #####################DESCOMENTAR
    #    print(f"      - {col}: {typ}")

    # 5) Construir DDL y crear tabla
    col_defs = [
        sql.SQL("{} {}").format(sql.Identifier(col), sql.SQL(col_types[col]))
        for col in df.columns
    ]

    print(f"   → Creando tabla: {table_name} ...")
    ddl = sql.SQL("CREATE TABLE {} ({});").format(
        sql.Identifier(table_name),
        sql.SQL(", ").join(col_defs)
    )
    cur.execute(ddl)
    conn.commit()
    print(f"   ✓ Tabla creada: {table_name}")

    cur.close()
    return df  # DataFrame ya normalizado

# ------------------------------------------------------------------------------------
# 2) Insertar datos del DataFrame en la tabla (en lotes)
# ------------------------------------------------------------------------------------
def insert_dataframe(conn, table_name: str, df: pd.DataFrame, batch_size: int = 2000):
    cur = conn.cursor()

    # Asegurar misma normalización que en la creación (por si se llama suelto)
    df = df.rename(columns={c: sanitize_identifier(str(c)) for c in df.columns})
    for col in df.columns:
        if pg_type_from_series(df[col]) == "BOOLEAN" and df[col].dtype != bool:
            df[col] = normalize_boolean_column(df[col])

    # NaN -> None (se insertan como NULL en Postgres)
    df = df.where(pd.notnull(df), None)
    cols = list(df.columns)

    
# === Mostrar por pantalla ANTES de insertar ===
    print(f"==> Preparando inserción en tabla: {table_name}")
    print(f"   → Orden de columnas: {cols}")
    total_rows = len(df)
    print(f"   → Total de filas a insertar: {total_rows}")

    placeholders = sql.SQL(", ").join(sql.Placeholder() for _ in cols)
    insert_stmt = sql.SQL("INSERT INTO {} ({}) VALUES ({})").format(
        sql.Identifier(table_name),
        sql.SQL(", ").join(sql.Identifier(c) for c in cols),
        placeholders
    )

    print(f"   → Insertando datos en: {table_name} (filas={len(df)}) ...")
    batch = []
    for row in df.itertuples(index=False, name=None):
        batch.append(row)
        if len(batch) >= batch_size:
            cur.executemany(insert_stmt.as_string(conn), batch)
            batch.clear()
    if batch:
        cur.executemany(insert_stmt.as_string(conn), batch)

    conn.commit()
    cur.close()
    print(f"   ✓ Datos insertados en: {table_name}")



In [130]:
# ------------------------------------------------------------------------------------
# Flujo principal: crea tabla + inserta datos de cada pestaña
# ------------------------------------------------------------------------------------
def create_structure_from_excel(
    excel_path,
    admin_db,
    db_prefix, db_name):
    """
    - Obtiene pestañas con mostrar_pestanas_excel()
    - Conecta SIEMPRE a 'db_name' (PGDATABASE, p. ej. 'reign')
    - Por cada pestaña:
        * Lee hoja (mensajes “en proceso”)
        * DROP TABLE IF EXISTS + CREATE TABLE
        * INSERT de todas las filas
        * SELECTs de validación
    """
    hojas = mostrar_pestanas_excel(excel_path, imprimir=True)
    print(f"Total de pestañas: {len(hojas)}")
    print(f"Usando base de datos: {db_name}")

    with connect(db_name) as conn:
        for hoja in hojas:
            print("=" * 80)
            print(f"[En proceso] Abriendo hoja: {hoja}")
            df = read_sheet_resilient(excel_path, hoja)
            print(f"   ✓ Hoja leída: {hoja} (filas={len(df)}, columnas={df.shape[1]})")

            table_name = sanitize_identifier(hoja)

            # 1) Crear tabla (DROP + CREATE)
            df_norm = create_table_from_df(conn, table_name, df, drop_before_create=True, cascade=True)

            
            # 2) Inserta datos
            insert_dataframe(conn, table_name, df_norm, batch_size=2000)
            
            # 3) Validación básica
            cur = conn.cursor()
            cur.execute(sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(table_name)))
            total = cur.fetchone()[0]
            cur.execute(sql.SQL("SELECT * FROM {} LIMIT 5").format(sql.Identifier(table_name)))
            sample = cur.fetchall()
            cur.close()

            print(f"   ✓ Validación: {table_name} tiene {total} filas.")
            for r in sample:
                print("     ·", r)


            print(f"[Hecho] {db_name}.{table_name} cargado y verificado.")


    print("=" * 80)
    print(f"Proceso completado a las {datetime.now(timezone.utc)}: estructura creada e inserciones realizadas.")

In [131]:
create_structure_from_excel(
excel_path,
    PGUSER,
    DB_PREFIX,
    db_name
)

Pestañas encontradas:
 - products_full
 - product_family
 - product_class
 - product_line
 - sales_rep
 - team_managers
 - stores
 - customers
 - all_sales
Total de pestañas: 9
Usando base de datos: reign
[En proceso] Abriendo hoja: products_full
   ✓ Hoja leída: products_full (filas=91, columnas=9)
==> Tabla objetivo: products_full
   → Borrando tabla si existe: products_full ...
   ✓ Tabla eliminada (si existía): products_full
   → Creando tabla: products_full ...
   ✓ Tabla creada: products_full
==> Preparando inserción en tabla: products_full
   → Orden de columnas: ['product_id', 'product_family', 'product_class', 'product_line', 'product_item', 'retail_price_es', 'retail_price_ita', 'sales_price', 'eur']
   → Total de filas a insertar: 91
   → Insertando datos en: products_full (filas=91) ...
   ✓ Datos insertados en: products_full
   ✓ Validación: products_full tiene 91 filas.
     · (200001, 1, 'ACC', 1, 'Fancy Mat Bag', 28.5, 29.925, 22.8, 'EUR')
     · (200002, 1, 'ACC', 1, '